# Demo: luồng gọi LLM của DAIL-SQL · V-SQL · V-SQL+

Notebook đi theo kịch bản demo trong `note.md`: **Phần 1** mở đầu, **Phần 2** từng phương pháp (input → output mỗi lần gọi),
**Phần 3** số lần gọi LLM, **Phần 4** chốt. Các ô "Chuẩn bị" chạy trước buổi demo.

Mặc định chạy 3 câu minh hoạ của kịch bản, đều là câu **V-SQL làm đúng còn DAIL-SQL làm sai** (V-SQL+ cũng đúng):

| uid | CSDL | Câu hỏi (rút gọn) | DAIL-SQL sai vì | V-SQL đúng nhờ |
|---|---|---|---|---|
| 259 | superhero | Phần trăm siêu anh hùng có alignment 'Bad', bao nhiêu do Marvel Comics xuất bản | Viết `superhero.alignment` (cột không tồn tại, chỉ có `alignment_id`) | `v_superhero` (đúng view Hình 3) có sẵn `alignment`, `publisher_name`, không cần JOIN |
| 421 | toxicology | Liệt kê các nguyên tố của phân tử "TR004" | JOIN thừa với `molecule` và lấy `element` từ nhầm bảng | `v_atom` có sẵn `molecule_id`, chỉ cần 1 bảng |
| 380 | card_games | Bao nhiêu lá bài trong bộ "Hauptset Zehnte Edition" do Adam Rex vẽ | JOIN sai khoá (`cards.uuid = set_translations.setCode`) | Nối đúng `cards.setCode = v_set_translations.setCode` |

## Phần 1. Mở đầu (khoảng 30 giây)

Cùng một câu hỏi và cùng một mô hình (nhiệt độ 0), notebook cho thấy **mỗi phương pháp gọi LLM mấy lần, mỗi lần gửi gì, nhận gì**,
rồi chấm kết quả bằng cách chạy SQL trên CSDL BIRD mini-dev 2025-07-04.

| Cấu hình | Các lần gọi LLM |
|---|---|
| **DAIL-SQL** | Vòng 1: `CREATE TABLE` + 7 ví dụ train giống câu hỏi nhất → SQL nháp. Vòng 2: 7 ví dụ chọn lại theo khung SQL nháp → SQL cuối (prompt trùng vòng 1 thì không tính thêm) |
| **V-SQL (đưa hết view)** | Giai đoạn 1 (Hình 4): mọi view của D′ (view dựng theo JOIN của SQL đáp án) + câu hỏi → dummy SQL trên view. Giai đoạn 2 (Hình 5): định nghĩa view + dummy SQL + bảng gốc → SQL trên bảng gốc |
| **V-SQL (LLM chọn view)** | Thêm 1 lần gọi trước giai đoạn 1: danh mục D′ (tên + cột) + câu hỏi → tên view/bảng cần dùng (**tự thêm**, paper không mô tả cách chọn). Sau đó như trên, chỉ với các đối tượng đã chọn |
| **V-SQL+ (bỏ bầu chọn)** | Giai đoạn 1 (prompt V-SQL+): view tự động đã chọn bằng truy hồi BM25 + giá trị khớp câu hỏi + evidence → dummy SQL. Nếu lỗi tên cột/bảng: sửa lỗi (tối đa 2 lần). Giai đoạn 2 (Hình 5) chỉ khi dummy SQL dùng view |

**Chi phí:** câu trả lời LLM được lưu trong `llm_cache.sqlite`, chạy lại không tốn thêm. Nếu máy đã có kết quả 500 câu
(`results_new/cmp_gpt-4o-mini_d2025`), notebook chép cache đó sang thư mục demo, nên demo với `gpt-4o-mini` **không tốn tiền**.
Không có cache: khoảng 0,005 USD/câu cho cả 4 cấu hình (gpt-4o-mini), tức dưới 0,03 USD cho 5 câu.

Chạy được trên Colab (tự tải mã nguồn và dữ liệu) hoặc trên máy. Trên máy: dùng môi trường `.venv` của repo (đã có spaCy,
sentence-transformers) và cài thêm `jupyter matplotlib pandas`:
`.venv/bin/pip install jupyter matplotlib pandas` rồi `.venv/bin/jupyter notebook notebooks/Demo_LuongGoiLLM_3PP.ipynb`.

## Chuẩn bị (chạy trước buổi demo)

In [ ]:
#@title 1. Cấu hình
#@markdown **Nguồn model**: `openai` = API OpenAI (cần `OPENAI_API_KEY` trong biến môi trường, file `.env` hoặc Colab Secrets);
#@markdown `ollama` = model chạy qua Ollama (trên Colab T4: notebook tự cài).
BACKEND = "openai"  #@param ["openai", "ollama"]
MODEL = "gpt-4o-mini"  #@param {type:"string"}
#@markdown **Số câu** (1–500). Quay video: 3. Muốn có số liệu tổng hợp: 50–500.
SO_CAU = 3  #@param {type:"integer"}
#@markdown Cách chọn câu:
#@markdown - `kich_ban`: các câu minh hoạ của kịch bản trước (259, 421, 380, 470, 427: V-SQL đúng, DAIL-SQL sai); SO_CAU > 5 thì phần
#@markdown   còn lại lấy ngẫu nhiên theo tỷ lệ độ khó;
#@markdown - `ngau_nhien_theo_do_kho`: ngẫu nhiên (theo SEED), giữ tỷ lệ dễ / trung bình / khó như bộ dữ liệu (148 / 250 / 102 trên 500);
#@markdown - `dau_tien`: SO_CAU câu đầu; `theo_uid`: đúng các uid ghi trong UIDS (bỏ qua SO_CAU).
CACH_CHON = "kich_ban"  #@param ["kich_ban", "ngau_nhien_theo_do_kho", "dau_tien", "theo_uid"]
SEED = 0  #@param {type:"integer"}
UIDS = ""  #@param {type:"string"}
#@markdown Chỉ lấy câu của các CSDL này (cách nhau bằng dấu phẩy, để trống = mọi CSDL), vd `superhero`.
LOC_DB = ""  #@param {type:"string"}
#@markdown **Cấu hình chạy**
DAIL_SQL = True  #@param {type:"boolean"}
VSQL_HET_VIEW = True  #@param {type:"boolean"}
VSQL_LLM_CHON = True  #@param {type:"boolean"}
VSQLPLUS_KHONG_BAU_CHON = True  #@param {type:"boolean"}
#@markdown Dùng lại cache của lần chạy 500 câu (nếu có) để demo không tốn tiền.
DUNG_CACHE_500 = True  #@param {type:"boolean"}
#@markdown Số câu đầu tiên hiện chi tiết từng lần gọi ở Phần 2 (các câu còn lại chỉ tính vào bảng tổng hợp ở Phần 3).
HIEN_CHI_TIET = 5  #@param {type:"integer"}
#@markdown Số ký tự tối đa hiển thị cho mỗi prompt (0 = hiện đủ). Prompt V-SQL có thể dài hơn 30.000 ký tự.
CAT_PROMPT = 6000  #@param {type:"integer"}
#@markdown Dừng nếu tiền gọi API thật (không tính phần lấy từ cache) vượt mức này, USD. Tính theo giá gpt-4o-mini.
GIOI_HAN_USD = 1.0  #@param {type:"number"}
SO_CAU = max(1, min(500, int(SO_CAU)))

CAU_HINH = [k for k, on in [("dailsql", DAIL_SQL), ("vsql_all", VSQL_HET_VIEW), ("vsql_llm", VSQL_LLM_CHON),
                            ("vsqlplus_novote", VSQLPLUS_KHONG_BAU_CHON)] if on]
TEN = {"dailsql": "DAIL-SQL", "vsql_all": "V-SQL (đưa hết view)", "vsql_llm": "V-SQL (LLM chọn view)",
       "vsqlplus_novote": "V-SQL+ (bỏ bầu chọn)"}
assert CAU_HINH, "Chọn ít nhất một cấu hình"
print("Model:", BACKEND, MODEL, "| cấu hình:", [TEN[k] for k in CAU_HINH])

In [ ]:
#@title 2. Mã nguồn, dữ liệu, API key (Colab: tự tải, ~3–5 phút lần đầu)
import os, sys, json, shutil, subprocess, hashlib, urllib.request
from pathlib import Path
TREN_COLAB = "google.colab" in sys.modules


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if TREN_COLAB:
    REPO = Path("/content/vsqlplus")
    if REPO.is_dir():
        sh(f"git -C {REPO} pull -q")
    else:
        sh(f"git clone -q --depth 1 https://github.com/chloenguyen167/vsqlplus.git {REPO}")
    sh('pip install -q "sql_metadata<3" sentence-transformers spacy')
    sh("python -m spacy download en_core_web_sm -q")
    DATA = Path("/content/data/minidev/MINIDEV")
    if not (DATA / "dev_databases").is_dir():
        os.makedirs("/content/data", exist_ok=True)
        sh("wget -q -O /content/data/minidev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip")
        sh('unzip -q -o /content/data/minidev.zip "minidev/MINIDEV/*" -d /content/data')
    TRAIN_DIR = Path("/content/data/bird_train")
    OUT = Path("/content/demo_luong_goi_llm")
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_compare.py").exists())
    DATA = REPO / "minidev/MINIDEV"
    TRAIN_DIR = REPO / "minidev/bird_train"
    OUT = REPO / "results_new/demo_luong_goi_llm"
os.chdir(REPO)
sys.path.insert(0, str(REPO))

DATA_JSON = DATA / "mini_dev_sqlite_2025-07-04.json"
if not DATA_JSON.exists():
    urllib.request.urlretrieve("https://huggingface.co/datasets/birdsql/bird_mini_dev/resolve/"
                               "f65faf4ae3b638c1fa6df1d3370c8d92c8366301/data/mini_dev_sqlite-00000-of-00001.json", DATA_JSON)
assert hashlib.md5(DATA_JSON.read_bytes()).hexdigest() == "595b6b014a528f4386d2e16f74d14b40", "Sai file câu hỏi 2025-07-04"

if "dailsql" in CAU_HINH:
    try:                                      # DAIL-SQL che tên schema bằng spaCy (thay CoreNLP); thiếu model -> ví dụ chọn khác
        import spacy; spacy.load("en_core_web_sm")
    except Exception:
        print("CẢNH BÁO: thiếu spaCy en_core_web_sm -> DAIL-SQL chọn ví dụ khác lần chạy 500 câu (không dùng được cache, "
              "kết quả khác). Cài: python -m spacy download en_core_web_sm")
    from vsql.baselines import fetch_bird_train
    if not (TRAIN_DIR / "train.json").exists():
        fetch_bird_train(str(TRAIN_DIR))      # chỉ lấy train.json + train_tables.json trong train.zip (HTTP Range)

OUT.mkdir(parents=True, exist_ok=True)
CACHE_500 = REPO / "results_new/cmp_gpt-4o-mini_d2025/llm_cache.sqlite"
if DUNG_CACHE_500 and BACKEND == "openai" and MODEL == "gpt-4o-mini" and CACHE_500.exists() \
        and not (OUT / "llm_cache.sqlite").exists():
    import sqlite3
    with sqlite3.connect(CACHE_500) as a, sqlite3.connect(OUT / "llm_cache.sqlite") as b:
        a.backup(b)                           # chép bản sao: demo không ghi vào kết quả 500 câu
    print("Đã chép cache 500 câu ->", OUT / "llm_cache.sqlite")

if BACKEND == "openai":
    from vsql.llm import find_api_key
    if not find_api_key():
        print("CHƯA CÓ OPENAI_API_KEY: chỉ chạy được các câu đã có trong cache (câu chưa có sẽ báo lỗi).")
print("Repo:", REPO, "| dữ liệu:", DATA, "| thư mục demo:", OUT)

In [ ]:
#@title 3. (Chỉ khi BACKEND = ollama trên Colab) Cài và khởi động Ollama, tải model
if BACKEND == "ollama" and TREN_COLAB:
    import time
    sh("apt-get -qq install -y zstd pciutils > /dev/null")
    sh("curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1")
    if not globals().get("ollama_proc") or ollama_proc.poll() is not None:
        ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                                       env=dict(os.environ, OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434"))
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
        except Exception:
            time.sleep(1)
    sh(f"ollama pull {MODEL}")
else:
    print("Bỏ qua (BACKEND =", BACKEND + ", Colab =", str(TREN_COLAB) + ")")

In [ ]:
#@title 4. Chuẩn bị: chọn câu, dựng view, client LLM có ghi nhật ký
import copy, time
from collections import Counter, defaultdict
import run_compare as RC
from vsql.methods import DbCtx, load_tables_json, run_item, score
from vsql.db import run_sql
from vsql.llm import OllamaLLM, OpenAICompatLLM

argv = ["--data", str(DATA_JSON), "--db-root", str(DATA / "dev_databases"), "--tables-json", str(DATA / "dev_tables.json"),
        "--backend", BACKEND, "--model", MODEL, "--out", str(OUT), "--no-embed",
        "--dail-train", str(TRAIN_DIR / "train.json"), "--dail-train-tables", str(TRAIN_DIR / "train_tables.json"),
        "--dail-cache", str(TRAIN_DIR)]
ARGS = RC.parse_args(argv)
data = RC.load_data(ARGS)
gold_by_db = defaultdict(list)
for it in data:
    gold_by_db[it["db_id"]].append(it["SQL"])     # view của V-SQL nhìn JOIN của mọi SQL đáp án trong CSDL (như paper, mục 6)

import random
pool = [it for it in data if not LOC_DB or it["db_id"] in LOC_DB.split(",")]
KICH_BAN = [259, 421, 380, 470, 427]


def theo_do_kho(pool, n, rnd, bo_qua=()):
    """n câu ngẫu nhiên, số câu mỗi độ khó tỷ lệ với số câu của độ khó đó trong pool (làm tròn theo phần dư lớn nhất)."""
    pool = [it for it in pool if it["uid"] not in set(bo_qua)]
    n = min(n, len(pool))
    nhom = defaultdict(list)
    for it in pool:
        nhom[it["difficulty"]].append(it)
    chia = {d: n * len(g) / len(pool) for d, g in nhom.items()}
    so = {d: int(x) for d, x in chia.items()}
    for d in sorted(chia, key=lambda d: (so[d] - chia[d], d))[:n - sum(so.values())]:
        so[d] += 1
    return sorted([it for d in sorted(nhom) for it in rnd.sample(nhom[d], so[d])], key=lambda it: it["uid"])


rnd = random.Random(SEED)
if CACH_CHON == "kich_ban":
    CAU = [data[u] for u in KICH_BAN[:SO_CAU]]
    if SO_CAU > len(KICH_BAN):
        CAU += theo_do_kho(pool, SO_CAU - len(KICH_BAN), rnd, KICH_BAN)
elif CACH_CHON == "theo_uid":
    CAU = [data[int(x)] for x in UIDS.replace(" ", "").split(",") if x]
elif CACH_CHON == "dau_tien":
    CAU = pool[:SO_CAU]
else:
    CAU = theo_do_kho(pool, SO_CAU, rnd)
assert CAU, "Không có câu nào (kiểm tra LOC_DB / UIDS)"
print(f"{len(CAU)} câu; độ khó:", dict(Counter(it["difficulty"] for it in CAU)),
      "| CSDL:", dict(Counter(it["db_id"] for it in CAU)))

tables_json = load_tables_json(str(DATA / "dev_tables.json"))
CTX = {}
for db in sorted({it["db_id"] for it in CAU}):
    CTX[db] = DbCtx(db, DATA / "dev_databases" / db, gold_by_db[db], ARGS, tables_json)
    print(f"{db}: {len(CTX[db].schema.tables)} bảng, {len(CTX[db].oracle)} view V-SQL (theo JOIN đáp án), "
          f"{len(CTX[db].auto)} view tự động V-SQL+")

if BACKEND == "openai":
    LLM = OpenAICompatLLM(MODEL, ARGS.base_url, None, str(OUT / "llm_cache.sqlite"), ARGS.num_predict)
else:
    LLM = OllamaLLM(MODEL, ARGS.host, str(OUT / "llm_cache.sqlite"), ARGS.num_ctx, ARGS.num_predict, ARGS.think)


GIA = (0.15e-6, 0.60e-6) if BACKEND == "openai" else (0.0, 0.0)     # USD / token vào, ra (gpt-4o-mini)
DA_TIEU = {"usd": 0.0, "goi_that": 0}


class GhiNhatKy:
    """Bọc client LLM: ghi lại mọi lần gọi (giai đoạn, prompt, câu trả lời, token, lấy từ cache hay gọi thật).
    giu_prompt=False (câu không hiện chi tiết): chỉ giữ một khoá băm của prompt, đỡ tốn bộ nhớ khi chạy nhiều câu."""

    def __init__(self, llm, giu_prompt=True):
        self.llm, self.log, self.giu = llm, [], giu_prompt

    def chat(self, prompt, meta=None, **opts):
        tu_cache = self.llm.cached(prompt, **opts) is not None
        if not tu_cache and DA_TIEU["usd"] > GIOI_HAN_USD:
            raise SystemExit(f"Đã tiêu ~{DA_TIEU['usd']:.2f} USD cho lần gọi thật, vượt GIOI_HAN_USD = {GIOI_HAN_USD}. "
                             "Tăng GIOI_HAN_USD rồi chạy lại ô này (phần đã gọi nằm trong cache, không tốn lại).")
        r = self.llm.chat(prompt, meta, **opts)
        if not tu_cache:
            DA_TIEU["goi_that"] += 1
            DA_TIEU["usd"] += r["prompt_tokens"] * GIA[0] + r["completion_tokens"] * GIA[1]
        self.log.append(dict(stage=(meta or {}).get("stage", "?"),
                             prompt=prompt if self.giu else hash(prompt), opts=opts, text=r["text"] if self.giu else "",
                             pt=r["prompt_tokens"], ct=r["completion_tokens"], sec=r["seconds"], tu_cache=tu_cache))
        return r


def args_cho(k):
    a = copy.copy(ARGS)
    a.vsql_select = {"vsql_all": "all", "vsql_llm": "llm"}.get(k, ARGS.vsql_select)
    return a


METHOD = {"dailsql": "dailsql", "vsql_all": "vsql", "vsql_llm": "vsql", "vsqlplus_novote": "vsqlplus_novote"}
print("Câu đầu tiên:", [(it["uid"], it["db_id"], it["difficulty"]) for it in CAU[:10]], "..." if len(CAU) > 10 else "")

In [ ]:
#@title 5. Chạy: mỗi câu × mỗi cấu hình
# (uid, cấu hình) -> dict(rec, log). Chạy lại ô này với cùng model thì bỏ qua các cặp đã xong; đổi model thì làm lại từ đầu.
if globals().get("KQ_CFG") != (BACKEND, MODEL):
    KQ = {}
KQ_CFG = (BACKEND, MODEL)
gold = {}
CHI_TIET = {it["uid"] for it in CAU[:HIEN_CHI_TIET]}
t_bat_dau = time.time()
for i, it in enumerate(CAU, 1):
    c = CTX[it["db_id"]]
    if it["uid"] not in gold:
        rows, err, sec = run_sql(c.path, it["SQL"], ARGS.timeout * 10)
        gold[it["uid"]] = (rows, sec)
    dong = []
    for k in CAU_HINH:
        if (it["uid"], k) not in KQ or (it["uid"] in CHI_TIET and isinstance(KQ[(it["uid"], k)]["log"][0]["prompt"], int)):
            g = GhiNhatKy(LLM, giu_prompt=it["uid"] in CHI_TIET)
            rec = run_item(METHOD[k], it, c, g, args_cho(k))
            KQ[(it["uid"], k)] = dict(rec=score(rec, *gold[it["uid"]], c.path, ARGS.timeout), log=g.log)
        x = KQ[(it["uid"], k)]
        dong.append(f"{TEN[k]} {'ĐÚNG' if x['rec']['correct'] else 'sai'} ({x['rec']['calls']} lần gọi)")
    if len(CAU) <= 20:
        print(f"uid {it['uid']:>3} [{it['difficulty']}]: " + " | ".join(dong))
    elif i % 25 == 0 or i == len(CAU):
        ex = " | ".join(f"{TEN[k]} {100 * sum(KQ[(u['uid'], k)]['rec']['correct'] for u in CAU[:i]) / i:.1f}%" for k in CAU_HINH)
        print(f"[{i}/{len(CAU)}] {time.time() - t_bat_dau:.0f}s · EX tạm thời: {ex}")
print(f"\nXong {len(CAU)} câu · {DA_TIEU['goi_that']} lần gọi API thật (~{DA_TIEU['usd']:.3f} USD), còn lại lấy từ cache.")

## Phần 2. Từng phương pháp: input → output mỗi lần gọi

Với mỗi câu: câu hỏi, evidence, SQL đáp án; rồi lần lượt 4 cấu hình. Mỗi lần gọi LLM là một dòng, bấm vào để mở **prompt gửi đi**
và **câu trả lời nhận về**. Gợi ý trình bày:
- **DAIL-SQL**: chỉ ra 7 ví dụ trong prompt và schema `CREATE TABLE`; vòng 2 khác vòng 1 ở bộ ví dụ.
- **V-SQL**: chỉ ra view trong schema giai đoạn 1 (ví dụ `v_superhero` có sẵn `alignment`, `publisher_name`), dummy SQL không có JOIN,
  rồi giai đoạn 2 khôi phục JOIN về bảng gốc. Bản LLM chọn view có thêm lần gọi chọn view ở đầu, prompt giai đoạn 1 ngắn hơn.
- **V-SQL+**: chỉ ra mục `# matched database values` và chú thích nguồn gốc cột trong `CREATE VIEW`.

In [ ]:
#@title 6. Xem từng lần gọi LLM: input → output (bấm vào từng dòng để mở prompt)
import html
from IPython.display import HTML, display

GIAI_DOAN = {
    "dail_pre": "Vòng 1 — 7 ví dụ chọn theo câu hỏi → SQL nháp",
    "dail_final": "Vòng 2 — 7 ví dụ chọn theo khung SQL nháp → SQL cuối",
    "vsql_sel": "Chọn view/bảng của D′ (LLM; tự thêm, không có trong paper)",
    "vsql_s1": "Giai đoạn 1 (Hình 4) — dummy SQL trên view",
    "vsql_s2": "Giai đoạn 2 (Hình 5) — khôi phục về bảng gốc",
    "plus_final": "Giai đoạn 1 (prompt V-SQL+) — dummy SQL trên view",
    "plus_fb0": "Sửa lỗi lần 1 — thêm bảng gốc vào schema",
    "plus_fb1": "Sửa lỗi lần 2 — thêm bảng gốc vào schema",
    "plus_s2": "Giai đoạn 2 (Hình 5) — khôi phục về bảng gốc",
}


def pre(s, cat=0):
    s = s or ""
    if cat and len(s) > cat:
        s = s[:cat] + f"\n\n… (đã cắt, còn {len(s) - cat:,} ký tự; đặt CAT_PROMPT = 0 để xem đủ)"
    return ("<pre style='white-space:pre-wrap;font-size:12px;max-height:420px;overflow:auto;"
            "background:#f6f6f4;padding:8px;border-radius:4px'>" + html.escape(s) + "</pre>")


def the_cau(it):
    out = [f"<h3>uid {it['uid']} · {it['db_id']} · {it['difficulty']}</h3>",
           f"<p><b>Câu hỏi:</b> {html.escape(it['question'])}<br><b>Evidence:</b> {html.escape(it.get('evidence') or '')}</p>",
           "<b>SQL đáp án</b>" + pre(it["SQL"])]
    for k in CAU_HINH:
        x = KQ[(it["uid"], k)]
        rec, log = x["rec"], x["log"]
        dau = "✅ đúng" if rec["correct"] else f"❌ sai ({rec['outcome']})"
        out.append(f"<h4>{TEN[k]} — {rec['calls']} lần gọi, {rec['prompt_tokens']:,} token vào, "
                   f"{rec['completion_tokens']:,} token ra — {dau}</h4>")
        info = []
        if rec.get("objects"):
            info.append("Đối tượng schema đưa vào prompt: <code>" + html.escape(", ".join(rec["objects"])) + "</code>")
        if rec.get("value_links"):
            info.append("Giá trị khớp câu hỏi: <code>" + html.escape(", ".join(f"{v!r} ∈ {t}.{c}" for v, t, c in rec["value_links"][:5])) + "</code>")
        if rec.get("views_used") is not None and k != "dailsql":
            info.append("View dummy SQL dùng: <code>" + html.escape(", ".join(rec["views_used"]) or "(không dùng view)") + "</code>")
        if rec.get("dail_examples"):
            info.append(f"{len(rec['dail_examples'])} ví dụ vòng 2, ví dụ đầu: <i>{html.escape(rec['dail_examples'][0])}…</i>")
        if info:
            out.append("<p style='font-size:13px'>" + "<br>".join(info) + "</p>")
        seen = set()
        for i, c in enumerate(log, 1):
            key = (c["prompt"], json.dumps(c["opts"], sort_keys=True))
            lap = key in seen
            seen.add(key)
            nhan = GIAI_DOAN.get(c["stage"], c["stage"])
            ghi = " · <i>prompt trùng lần trước, không tính thêm</i>" if lap else ""
            nguon = "cache" if c["tu_cache"] else "gọi thật"
            out.append(f"<details><summary><b>Lần {i}. {html.escape(nhan)}</b> — {c['pt']:,} token vào → {c['ct']:,} token ra "
                       f"({nguon}){ghi}</summary><b>Input (prompt, {len(c['prompt']):,} ký tự)</b>{pre(c['prompt'], CAT_PROMPT)}"
                       f"<b>Output</b>{pre(c['text'])}</details>")
        out.append("<b>SQL cuối</b>" + pre(rec["final_sql"]))
    return "\n".join(out)


for it in CAU[:HIEN_CHI_TIET]:
    display(HTML(the_cau(it) + "<hr>"))
if len(CAU) > HIEN_CHI_TIET:
    print(f"(Chỉ hiện chi tiết {HIEN_CHI_TIET}/{len(CAU)} câu đầu; tăng HIEN_CHI_TIET ở ô 1 rồi chạy lại ô 5–6 để xem thêm.)")

## Phần 3. Số lần gọi LLM

In [ ]:
#@title 7. Tổng hợp: số lần gọi LLM theo giai đoạn, token, độ chính xác trên các câu demo
import matplotlib.pyplot as plt
import pandas as pd

NHOM = {"vsql_sel": "Chọn view (LLM)", "dail_pre": "Giai đoạn 1 / vòng 1", "vsql_s1": "Giai đoạn 1 / vòng 1",
        "plus_final": "Giai đoạn 1 / vòng 1", "plus_fb0": "Sửa lỗi", "plus_fb1": "Sửa lỗi",
        "dail_final": "Giai đoạn 2 / vòng 2", "vsql_s2": "Giai đoạn 2 / vòng 2", "plus_s2": "Giai đoạn 2 / vòng 2"}
THU_TU = ["Chọn view (LLM)", "Giai đoạn 1 / vòng 1", "Sửa lỗi", "Giai đoạn 2 / vòng 2"]
MAU = dict(zip(THU_TU, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))   # 4 màu đầu bảng phân loại, đã kiểm tra mù màu

n = len(CAU)
dem = {k: Counter() for k in CAU_HINH}
bang = []
for k in CAU_HINH:
    for it in CAU:
        x = KQ[(it["uid"], k)]
        seen = set()
        for c in x["log"]:
            key = (c["prompt"], json.dumps(c["opts"], sort_keys=True))
            if key not in seen:                      # cùng cách đếm với chương trình chính: prompt trùng không tính
                seen.add(key)
                dem[k][NHOM.get(c["stage"], "Giai đoạn 1 / vòng 1")] += 1
    recs = [KQ[(it["uid"], k)]["rec"] for it in CAU]
    tin = sum(r["prompt_tokens"] for r in recs) / n
    tout = sum(r["completion_tokens"] for r in recs) / n
    bang.append({"Cấu hình": TEN[k], "Gọi LLM / câu": round(sum(r["calls"] for r in recs) / n, 2),
                 "Token vào / câu": round(tin), "Token ra / câu": round(tout),
                 "USD / câu (gpt-4o-mini)": round(tin * 0.15e-6 + tout * 0.6e-6, 5),
                 "Đúng": f"{sum(r['correct'] for r in recs)}/{n}"})
display(pd.DataFrame(bang).set_index("Cấu hình"))

# độ chính xác theo độ khó (có ý nghĩa khi chạy nhiều câu)
DO_KHO = ["simple", "moderate", "challenging"]
theo_kho = []
for k in CAU_HINH:
    dong = {"Cấu hình": TEN[k]}
    for d in DO_KHO + ["tổng"]:
        us = [it for it in CAU if d == "tổng" or it["difficulty"] == d]
        if us:
            dong[f"{d} (n={len(us)})"] = round(100 * sum(KQ[(it["uid"], k)]["rec"]["correct"] for it in us) / len(us), 1)
    theo_kho.append(dong)
print("EX (%) theo độ khó:")
display(pd.DataFrame(theo_kho).set_index("Cấu hình"))

fig, ax = plt.subplots(figsize=(8, 0.6 * len(CAU_HINH) + 1.2))
ys = list(range(len(CAU_HINH)))[::-1]
left = [0.0] * len(CAU_HINH)
for g in THU_TU:
    w = [dem[k][g] / n for k in CAU_HINH]
    if not any(w):
        continue
    ax.barh(ys, w, left=left, height=0.5, color=MAU[g], edgecolor="white", linewidth=2, label=g)
    left = [a + b for a, b in zip(left, w)]
for y, tot in zip(ys, left):
    ax.text(tot + 0.05, y, f"{tot:.2f}", va="center", fontsize=9, color="#52514e")
ax.set_yticks(ys, [TEN[k] for k in CAU_HINH])
ax.set_xlabel(f"Số lần gọi LLM trung bình mỗi câu ({n} câu demo)")
ax.set_xlim(0, max(left) + 0.6)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8, ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0))
plt.tight_layout()
plt.show()

### Số đo trên đủ 500 câu (gpt-4o-mini, `results_new/cmp_gpt-4o-mini_d2025`)

| Cấu hình | EX (%) | Riêng giai đoạn 1 (%) | Gọi LLM / câu | Token vào / câu | Token ra / câu | USD / 500 câu |
|---|---|---|---|---|---|---|
| DAIL-SQL | 50.0 | 49.6 | 1.75 | 2,946 | 479 | 0.36 |
| V-SQL (đưa hết view) | 46.0 | 45.2 | 2.00 | 14,412 | 129 | 1.12 |
| V-SQL (LLM chọn view) | 46.8 | 43.2 | 3.00 | 8,219 | 142 | 0.66 |
| V-SQL+ (bỏ bầu chọn) | 52.6 | 53.4 | 1.93 | 5,984 | 105 | 0.48 |

- **Số lần gọi không quyết định chi phí hay độ chính xác.** V-SQL đưa hết view chỉ gọi 2 lần nhưng prompt dài nhất (14k token,
  đắt nhất). DAIL-SQL đọc ít nhất nhưng sinh nhiều token nhất (479 token/câu, vì mô hình viết tiếp sau `SELECT `).
- V-SQL+ (bỏ bầu chọn): giai đoạn 1 luôn 1 lần, sửa lỗi 0,05 lần/câu, giai đoạn 2 chỉ chạy ở 88% số câu (dummy SQL có dùng view).
- DAIL-SQL: 124/500 câu chỉ 1 lần gọi vì prompt vòng 2 trùng hệt vòng 1 (cùng 7 ví dụ).
- Bước LLM chọn view của V-SQL là lần gọi tự thêm (paper không mô tả). Nó làm prompt giai đoạn 1 ngắn đi khoảng 43%, nhưng độ
  chính xác chỉ hơn bản đưa hết view 0,8 điểm (không có ý nghĩa thống kê).

## Phần 4. Chốt (khoảng 30 giây)

- **View giúp đúng chỗ paper nói:** ở các câu demo, DAIL-SQL sai vì tự JOIN hoặc đoán tên cột, còn V-SQL hỏi thẳng trên view nên không
  cần JOIN. Nhưng trên đủ 500 câu, V-SQL vẫn thua DAIL-SQL 3,2 điểm (46,8 so với 50,0, không có ý nghĩa thống kê): V-SQL đúng riêng 51 câu,
  DAIL-SQL đúng riêng 67 câu. Phân tích ở `docs/NHAN_XET_KET_QUA_GPT4OMINI.md`.
- **Số lần gọi không quyết định chi phí.** V-SQL đưa hết view chỉ gọi 2 lần nhưng đắt nhất vì prompt dài nhất. DAIL-SQL đọc ít nhất
  nhưng viết nhiều nhất.
- **V-SQL+ (bỏ bầu chọn) đúng nhất (52,6), chi phí chỉ hơn DAIL-SQL một chút.** Nó thắng nhờ gần như không có lỗi thực thi (sai tên cột,
  hàm ngoài SQLite), dù vẫn hiểu câu hỏi kém DAIL-SQL.
- **Công bằng với paper:** bước LLM chọn view của V-SQL là phần tự thêm. Bầu chọn của V-SQL+ đã bị bỏ để cùng điều kiện 1 mẫu với
  DAIL-SQL.